# Lab 2 (advanced) — Data quality and preparation
**Data Analysis — Master AI / Master RSD** · Chapters 1–2: data exploration and preparation

An Algerian e-commerce company hands you two **raw** extracts from its system:
- `clients_raw.csv` — a **structured** but dirty file (duplicates, wrong types, impossible values, inconsistent categories…);
- `purchases.json` — a **semi-structured** file: a list of purchases nested inside each client.

**Mission:** produce a dataset that is **clean, enriched and ready for modelling**, with no data leakage.

| Part | Topic | Concepts |
|---|---|---|
| 1 | Multi-source import | `read_csv`, `json_normalize` |
| 2 | Quality audit | duplicates, types, missing values, categories |
| 3 | Cleaning | type conversion, text normalisation, dates |
| 4 | Outliers | IQR, z-score, sentinel codes, log transform |
| 5 | Missing values | MCAR/MAR/MNAR, group imputation, indicator |
| 6 | Feature engineering | tenure, RFM, `merge`, `pd.cut` |
| 7 | Exploration | correlations, `pivot_table`, `crosstab` |
| 8 | Preparing for modelling | `train_test_split`, `ColumnTransformer`, data leakage |
| 9 | Challenge | a reproducible `clean()` function |

> Work in pairs. Fill in the **`# TODO`** cells. The **✅ Self-check** cells validate your results: if one raises an error, fix the previous cell before moving on — each part depends on the one before.

---
## Setup

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
REF_DATE = pd.Timestamp("2026-01-01")      # reference date of the analysis

---
## Part 1 — Multi-source import
### 1.1 The structured file
Load `clients_raw.csv` into `df_raw`. **Never modify `df_raw`**: we work on a copy so we can always go back to the source.

In [ ]:
df_raw = ...   # TODO

df_raw.head()

✅ **Self-check**

In [ ]:
assert df_raw.shape == (1025, 8), f"Expected (1025, 8), got {df_raw.shape}"
print("✅ 1.1 correct")

### 1.2 The semi-structured file
`purchases.json` is a **list of clients**, each carrying a **list of purchases**. First look at its raw shape, then "flatten" it into a table with **one row per purchase** using `pd.json_normalize(data, record_path=..., meta=...)`.

In [ ]:
with open("purchases.json", encoding="utf-8") as f:
    data = json.load(f)

data[0]

In [ ]:
purchases = ...   # TODO: json_normalize, one row per purchase, with the client_id column

purchases.head()

✅ **Self-check**

In [ ]:
assert list(purchases.columns) == ["date", "amount", "category", "client_id"], "Expected columns: date, amount, category, client_id"
assert len(purchases) == 5160, "Wrong number of purchases"
assert purchases["client_id"].nunique() == 900, "900 clients made at least one purchase"
print("✅ 1.2 correct")

---
## Part 2 — Quality audit
### 2.1 A reusable audit function
Write `audit(df)` returning a DataFrame with **one row per column** of `df` and the columns:
`dtype`, `missing` (number of NaN), `pct_missing` (in %, rounded to 0.1), `n_unique` (number of distinct values).

In [ ]:
def audit(df):
    # TODO
    ...

audit(df_raw)

✅ **Self-check**

In [ ]:
a = audit(df_raw)
assert list(a.columns) == ["dtype", "missing", "pct_missing", "n_unique"]
assert a.loc["income", "dtype"] == "object", "income is read as text: expected at this stage"
assert a.loc["city", "missing"] == 12
print("✅ 2.1 correct")

### 2.2 Reading the audit
1. How many rows are **exact duplicates**? How many distinct `client_id`?
2. Why is `income` of type `object`? Display a few values.
3. How many distinct **categories** for `city` and `satisfaction`? How many should there be?
4. Which `age` values are impossible?

In [ ]:
n_duplicates = ...   # TODO
print(n_duplicates, df_raw["client_id"].nunique())

# TODO: a few income values, value_counts of city and satisfaction, describe of age


✅ **Self-check**

In [ ]:
assert n_duplicates == 25, "25 duplicated rows expected"
print("✅ 2.2 correct")

✍️ **Your findings:** …

---
## Part 3 — Cleaning
Work on `df = df_raw.copy()`. In order:
1. drop **duplicates** and reset the index;
2. convert `income` to **numeric** (remove the commas, then `pd.to_numeric(..., errors="coerce")`);
3. normalise `city`: no surrounding spaces, **Title Case** (`str.strip().str.title()`);
4. normalise `satisfaction`: no spaces, lower case;
5. convert `signup_date` to `datetime` (two mixed formats: `format="mixed", dayfirst=True`);
6. replace ages outside **[18, 100]** with `NaN`.

In [ ]:
df = df_raw.copy()

# 1. duplicates
# TODO

# 2. income -> numeric
# TODO

# 3. city
# TODO

# 4. satisfaction
# TODO

# 5. signup_date
# TODO

# 6. impossible ages
# TODO

audit(df)

✅ **Self-check**

In [ ]:
assert df.shape == (1000, 8), "1000 rows after removing duplicates"
assert df["income"].dtype == "float64", "income must be numeric"
assert df["income"].isna().sum() == 40, "40 blank incomes expected"
assert sorted(df["city"].dropna().unique()) == ["Algiers", "Annaba", "Constantine", "Oran", "Tlemcen"]
assert sorted(df["satisfaction"].unique()) == ["high", "low", "medium"]
assert pd.api.types.is_datetime64_any_dtype(df["signup_date"]) and df["signup_date"].notna().all()
assert df["age"].isna().sum() == 8 and df["age"].between(18, 100).sum() == 992
assert df_raw.shape == (1025, 8), "df_raw must not have been modified!"
print("✅ Part 3 correct")

---
## Part 4 — Outliers
### 4.1 Two detection rules
On `income`:
- **IQR rule**: outlier if $x < Q_1 - 1.5\,IQR$ or $x > Q_3 + 1.5\,IQR$;
- **z-score rule**: outlier if $|z| > 3$ with $z = (x - \bar x)/\sigma$.

Count the incomes flagged by each rule.

In [ ]:
q1, q3 = ...   # TODO
iqr = q3 - q1
outlier_iqr = ...   # TODO: boolean Series
z = ...             # TODO
outlier_z = ...     # TODO

print("IQR:", outlier_iqr.sum(), "| z-score:", outlier_z.sum())

✅ **Self-check**

In [ ]:
assert outlier_iqr.sum() == 42 and outlier_z.sum() == 9
print("✅ 4.1 correct")

In [ ]:
df.loc[outlier_iqr, "income"].sort_values(ascending=False).value_counts(sort=False).head(15)

### 4.2 Outlier ≠ error
The two rules disagree, and **neither** tells an error apart from a genuine extreme case.
1. One value appears **exactly** several times at the very top: which one? What does it probably represent?
2. Replace this **sentinel code** with `NaN` (it is a disguised missing value), but **keep** the genuine high earners.

In [ ]:
n_codes = ...   # TODO: number of incomes equal to the code
# TODO: replace the code with NaN


✅ **Self-check**

In [ ]:
assert n_codes == 3 and df["income"].isna().sum() == 43 and df["income"].max() < 999999
print("✅ 4.2 correct")

### 4.3 Skewness and log transform
Income is **right-skewed**: the IQR rule therefore flags many perfectly normal clients. Create `income_log = np.log1p(income)`, compare the **skewness** (`.skew()`) before/after, plot both histograms, then recount IQR outliers **on the log scale**.

In [ ]:
# TODO: df["income_log"] = ...
# TODO: skew before / after
# TODO: two histograms side by side
# TODO: IQR outliers on income_log


**Checkpoint**: `df_clean` is the result of cleaning (Parts 3–4). We reuse it in the final challenge.

In [ ]:
df_clean = df.copy()

---
## Part 5 — Missing values
### 5.1 Is missingness random?
We distinguish:
- **MCAR** (*Missing Completely At Random*): missingness depends on nothing;
- **MAR** (*At Random*): it depends on **other** observed variables (e.g. the segment);
- **MNAR** (*Not At Random*): it depends on the value **itself** (e.g. high earners refuse to answer).

Compute the **rate of missing income per segment**. Does missingness seem linked to the segment?

In [ ]:
# TODO: rate of missing income per segment


### 5.2 Imputing (for descriptive analysis)
On a **copy** `df_desc`:
1. add an indicator `income_missing` (0/1) **before** imputing — the absence of information is information;
2. impute `income` with the **median of its segment** (`groupby(...).transform("median")`);
3. compare with imputation by the **global median**: which preserves the differences between segments better?

In [ ]:
df_desc = df.copy()
# TODO


✅ **Self-check**

In [ ]:
assert df_desc["income"].isna().sum() == 0 and df_desc["income_missing"].sum() == 43
assert df["income"].isna().sum() == 43, "df must not be imputed (work on df_desc)"
print("✅ 5.2 correct")

⚠️ **Trick question**: if the goal is later to **predict the segment**, can we use this segment-based imputation in the model? Answer before Part 8.

✍️ …

---
## Part 6 — Feature engineering
### 6.1 Features derived from clients
- `tenure_years`: years between `signup_date` and `REF_DATE` (rounded to 0.1);
- `age_band`: `pd.cut` of age into `[18-25)`, `[25-35)`, `[35-50)`, `[50-65)`, `65+` (left edges included: `right=False`).

In [ ]:
# TODO: df["tenure_years"] = ...
# TODO: df["age_band"] = pd.cut(...)


✅ **Self-check**

In [ ]:
assert df["tenure_years"].between(0, 11.1).all()
assert df["age_band"].isna().sum() == 8, "only the 8 unknown ages have no band"
print("✅ 6.1 correct")

### 6.2 RFM indicators from purchases
**RFM** is a marketing classic:
- **R**ecency: days between the **last** purchase and `REF_DATE`;
- **F**requency: number of purchases;
- **M**onetary: total amount spent.

1. Convert `purchases["date"]` to datetime, compute `rfm` (one row per client) with `groupby(...).agg(...)`.
2. Merge into `df` with `merge(..., how="left", validate="one_to_one")`.
3. The 100 clients **without purchases** have NaN: what value makes sense for `frequency` and `monetary`? And for `recency`?

In [ ]:
# TODO


✅ **Self-check**

In [ ]:
assert len(df) == 1000, "the merge must neither lose nor duplicate clients"
assert df["frequency"].sum() == len(purchases) and (df["frequency"] == 0).sum() == 100
assert df["monetary"].sum() == purchases["amount"].sum()
print("✅ 6.2 correct")

---
## Part 7 — Exploration
1. Plot the **correlation matrix** of the numeric variables (`sns.heatmap`, `annot=True`).
2. Pivot table of **average monetary** by `segment` × `education` (`pivot_table`).
3. Distribution of `satisfaction` **in %** within each segment (`pd.crosstab(..., normalize="index")`).

Write **three findings** with numbers.

In [ ]:
# TODO


✍️ **Your three findings:**
1. …
2. …
3. …

---
## Part 8 — Preparing the data for a model
**Goal**: prepare `X` (features) and `y = segment` (target) for a future classification model.

Golden rule: **anything learned from the data** (imputation median, scaling mean/std, one-hot categories) **is learned on the training set only**, then applied to the test set. Otherwise the test set "leaks" into training and the measured performance is over-optimistic.

1. Split `X` / `y`, then `train_test_split` (20 % test, `stratify=y`, `random_state=0`).
2. Build a `ColumnTransformer`:
   - **numeric** `age, income_log, tenure_years, recency, frequency, monetary` → `SimpleImputer(median)` + `StandardScaler`;
   - **ordinal** `education, satisfaction` → `SimpleImputer(most_frequent)` + `OrdinalEncoder` **with the explicit order**;
   - **nominal** `city` → `SimpleImputer(most_frequent)` + `OneHotEncoder(handle_unknown="ignore")`.
3. `fit_transform` on train, `transform` on test.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

num_cols = ["age", "income_log", "tenure_years", "recency", "frequency", "monetary"]
ord_cols = ["education", "satisfaction"]
nom_cols = ["city"]

In [ ]:
X = ...   # TODO
y = ...   # TODO
X_train, X_test, y_train, y_test = ...   # TODO

prep = ColumnTransformer([
    # TODO: ("num", ..., num_cols), ("ord", ..., ord_cols), ("nom", ..., nom_cols)
])

X_train_p = ...   # TODO
X_test_p  = ...   # TODO
print(X_train_p.shape, X_test_p.shape)

✅ **Self-check**

In [ ]:
assert X_train_p.shape == (800, 13) and X_test_p.shape == (200, 13), "6 numeric + 2 ordinal + 5 cities = 13 columns"
assert not np.isnan(X_train_p).any() and not np.isnan(X_test_p).any()
assert abs(X_train_p[:, 0].mean()) < 1e-9, "the scaler must be fitted on train"
print("✅ Part 8 correct")

In [ ]:
pd.DataFrame(X_train_p, columns=prep.get_feature_names_out()).head()

### 8.2 Bonus — a first model
Chain `prep` and a `LogisticRegression` in a single `Pipeline`, train on the training set and report **accuracy** on the test set. (Models come later in the course: the point is to see that preparation plugs into the model.)

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO


---
## Part 9 — Challenge: reproducible cleaning
In a real project, raw data **comes back every month**. Gather Parts 3 and 4 into a function `clean(df_raw)` that returns a new DataFrame **identical to `df_clean`**, without modifying its argument. Then export the result as `clients_clean.csv`.

In [ ]:
def clean(raw):
    # TODO
    ...

✅ **Self-check**

In [ ]:
copy = df_raw.copy()
pd.testing.assert_frame_equal(clean(df_raw), df_clean)
pd.testing.assert_frame_equal(df_raw, copy)   # argument not modified
print("✅ Challenge passed: reproducible cleaning")

In [ ]:
clean(df_raw).to_csv("clients_clean.csv", index=False, encoding="utf-8")

---
## Summary
| Problem | Detection | Treatment |
|---|---|---|
| Duplicates | `duplicated()` | `drop_duplicates()` |
| Number stored as text | `dtypes` = `object` | `str.replace` + `to_numeric(errors="coerce")` |
| Inconsistent categories | `value_counts()` | `str.strip().str.lower()/.title()` + `replace` |
| Mixed date formats | inspection | `to_datetime(format="mixed", dayfirst=True)` |
| Impossible values | `describe()`, business bounds | → `NaN` |
| Sentinel codes | repeated extreme value | → `NaN` |
| Statistical outliers | IQR (on log), z-score | **investigate**, keep if plausible |
| Missing values | `isna()`, rate per group | indicator + imputation **inside the pipeline** |
| Data leakage | — | `fit` on train only, never use the target |

> **Key takeaway:** cleaning means **deciding** — and every decision must be justified, traced and reproducible.